<a href="https://colab.research.google.com/github/trevorphix001/AI-SDR-Sales-Development-Representative-/blob/main/AI_SDR_Nemotron1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI SDR — 3-Agent Sales Development Rep

Three specialized agents, powered by **NVIDIA Nemotron 3.5 Lightning** (via NVIDIA's hosted,
OpenAI-compatible API), collaborate to turn basic prospect info into a personalized B2B sales
email:

```
User Input → 1. Prospect Research → 2. Company Intelligence → 3. Email Writer → Personalized Email
```

This is a **sequential pipeline**, not a chatbot. Each agent has one job, a fixed input, and a fixed
structured output (a Pydantic model). The Python pipeline — not the model — controls the flow
between agents.

All inference happens through NVIDIA's hosted API — nothing is downloaded or run locally.

**Before you run this notebook:** add your NVIDIA API key as a Colab secret named
`NVIDIA_API_KEY` (key icon in the left sidebar → *Secrets* → *Add new secret*). Get a key at
[build.nvidia.com](https://build.nvidia.com/). Never paste the key directly into a cell.


## 01. Install Dependencies

In [ ]:
# NVIDIA's hosted API is OpenAI-compatible, so we talk to it with the
# standard OpenAI Python SDK, just pointed at a different base_url.
!pip install -q -U openai pydantic gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 3.2 MB/s eta 0:00:00


## 02. Configure NVIDIA API

Reads `NVIDIA_API_KEY` from Colab Secrets. If you're running this outside Colab (e.g. locally), it
falls back to an environment variable, then to a hidden prompt — the key is never printed or
written into the notebook itself.


In [ ]:
import os

NVIDIA_API_KEY = None

# 1. Colab Secrets (preferred — this is what the assignment asks for)
try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except Exception:
    pass  # not running in Colab, or secret not set

# 2. Environment variable (useful if you run this notebook locally / in CI)
if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY")

# 3. Last resort: hidden interactive prompt (input is masked, never echoed or printed)
if not NVIDIA_API_KEY:
    import getpass
    print("NVIDIA_API_KEY not found in Colab Secrets or environment variables.")
    NVIDIA_API_KEY = getpass.getpass("Paste your NVIDIA API key (input hidden): ").strip()

if not NVIDIA_API_KEY:
    raise RuntimeError(
        "NVIDIA API key not found. In Colab: click the key icon in the left sidebar -> "
        "'Secrets' -> 'Add new secret' -> name it NVIDIA_API_KEY -> toggle 'Notebook access' "
        "on, then re-run this cell."
    )

from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)
print("NVIDIA API client configured.")  # never print the key itself


NVIDIA API client configured.


## 03. Imports

In [ ]:
import json
import time
import traceback
from typing import Optional, Type, TypeVar

import openai
from pydantic import BaseModel, ValidationError


## 04. Configuration

Single place to change the model used everywhere in this notebook.


In [ ]:
# Keep the model name in one place so it's easy to change later.
MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"

# Retry behaviour for transient errors (rate limits, brief outages)
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 2  # doubles each retry: 2s, 4s, 8s

# How many times to re-ask the model if it returns JSON that doesn't parse
# or doesn't match the expected schema.
MAX_JSON_RETRIES = 2

UNKNOWN = "Unknown / Not provided"


## 05. Pydantic Models

Structured contracts passed *between* agents — never raw strings.


In [ ]:
from pydantic import BaseModel


class ProspectResearch(BaseModel):
    prospect_name: str
    job_title: str
    company: str
    role_summary: str
    relevant_context: list[str]
    personalization_points: list[str]
    potential_pain_points: list[str]


class CompanyIntelligence(BaseModel):
    company_name: str
    industry: str
    company_summary: str
    products_or_services: list[str]
    target_market: str
    potential_challenges: list[str]
    potential_use_cases: list[str]


class SalesEmail(BaseModel):
    subject: str
    body: str

## 06. Nemotron Helper

`call_nemotron(system_prompt, user_prompt)` is the single, shared entry point to the model — every
agent goes through it, so all API-call logic (retries, error translation, never leaking the key)
lives in one place.

For the two research agents and the email writer, we need *structured* (Pydantic-shaped) output.
Nemotron is called through the plain chat-completions endpoint (not every OpenAI-compatible NIM
model supports enforced JSON schemas), so `call_nemotron_structured(...)` asks for JSON in the
prompt itself and validates what comes back with Pydantic. If the response is malformed — bad JSON,
or JSON that doesn't match the schema — it retries by sending the parse error back to the model and
asking it to correct itself, up to `MAX_JSON_RETRIES` times, before giving up with a clear error.


In [ ]:
import json
import time
from typing import Type, TypeVar
import openai
from pydantic import BaseModel, ValidationError


class AgentError(Exception):
    """Raised when an agent cannot produce a usable result after retries."""
    pass


T = TypeVar("T", bound=BaseModel)


def _translate_openai_error(e: Exception) -> AgentError:
    """Turns an openai SDK exception into a short, understandable AgentError.
    Never includes the API key or raw request/response bodies."""
    if isinstance(e, openai.AuthenticationError):
        return AgentError(
            "NVIDIA API key was rejected (missing, invalid, or expired). "
            "Check the NVIDIA_API_KEY Colab secret."
        )
    if isinstance(e, openai.PermissionDeniedError):
        return AgentError("NVIDIA API denied this request (insufficient permissions for this model/key).")
    if isinstance(e, openai.NotFoundError):
        return AgentError(f"Model '{MODEL}' was not found on NVIDIA's hosted API. Check the MODEL name.")
    if isinstance(e, openai.RateLimitError):
        return AgentError("NVIDIA API rate limit hit repeatedly. Wait a moment and try again.")
    if isinstance(e, openai.APITimeoutError):
        return AgentError("The request to NVIDIA's API timed out. Try again in a moment.")
    if isinstance(e, openai.BadRequestError):
        return AgentError(f"NVIDIA API rejected the request as invalid: {e}")
    if isinstance(e, openai.APIConnectionError):
        return AgentError("Could not reach NVIDIA's hosted API. Check your internet connection.")
    if isinstance(e, openai.APIStatusError):
        return AgentError(f"NVIDIA API error (status {e.status_code}).")
    return AgentError(f"Unexpected error calling NVIDIA API: {e}")


def _is_transient(e: Exception) -> bool:
    if isinstance(e, (openai.RateLimitError, openai.APITimeoutError, openai.APIConnectionError)):
        return True
    if isinstance(e, openai.InternalServerError):
        return True
    return False


def call_nemotron(system_prompt: str, user_prompt: str) -> str:
    """
    Sends a system + user message to Nemotron 3.5 Lightning via NVIDIA's hosted API
    and returns the response text. Retries transient errors (rate limits,
    brief outages) with backoff; raises a clear AgentError otherwise.
    """
    if not user_prompt or not user_prompt.strip():
        raise AgentError("Empty prompt passed to call_nemotron.")

    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,
            )
            text = (response.choices[0].message.content or "").strip()
            if not text:
                raise AgentError("Nemotron returned an empty response.")
            return text
        except AgentError:
            raise
        except Exception as e:
            last_error = e
            if _is_transient(e) and attempt < MAX_RETRIES - 1:
                wait = RETRY_BACKOFF_SECONDS * (2 ** attempt)
                print(f"  Transient error ({type(e).__name__}). Retrying in {wait}s... "
                      f"[attempt {attempt + 1}/{MAX_RETRIES}]")
                time.sleep(wait)
                continue
            raise _translate_openai_error(e)

    raise AgentError(f"Nemotron call failed after {MAX_RETRIES} attempt(s): {last_error}")


def _extract_json(text: str) -> str:
    """Best-effort extraction of a JSON object from model text output (strips
    markdown code fences and any leading/trailing prose)."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        if text.lstrip().lower().startswith("json"):
            text = text.lstrip()[4:]
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return text
    return text[start:end + 1]


def call_nemotron_structured(system_prompt: str, user_prompt: str, response_schema: Type[T]) -> T:
    """
    Calls call_nemotron() and parses/validates the result as response_schema.
    If the output is malformed (bad JSON, or doesn't match the schema), sends
    the error back to the model and asks it to correct itself, up to
    MAX_JSON_RETRIES times.
    """
    schema_hint = json.dumps(response_schema.model_json_schema(), indent=2)
    prompt = f"""{user_prompt}

Respond with ONLY a single JSON object (no markdown, no commentary) matching this JSON Schema:
{schema_hint}
"""

    last_error = None
    for attempt in range(MAX_JSON_RETRIES + 1):
        raw_text = call_nemotron(system_prompt, prompt)
        try:
            return response_schema.model_validate_json(_extract_json(raw_text))
        except (json.JSONDecodeError, ValidationError) as e:
            last_error = e
            if attempt < MAX_JSON_RETRIES:
                print(f"  Model output did not match schema. Asking it to correct itself... "
                      f"[attempt {attempt + 1}/{MAX_JSON_RETRIES}]")
                prompt = f"""{user_prompt}

Your previous response could not be parsed as valid JSON matching the schema ({e}).
Respond again with ONLY a single, valid JSON object (no markdown, no commentary) matching this JSON Schema:
{schema_hint}
"""

    raise AgentError(f"Model output did not match the expected schema after retries: {last_error}")

## 07. Prospect Research Agent

**Role:** understand the individual prospect and surface genuine personalization angles.

This MVP does not implement web scraping or search. The agent works only from the info you
supply, plus reasonable general knowledge about the role — it must not invent specific facts.
Anything it can't determine is reported as `"Unknown / Not provided"` rather than guessed at.


In [ ]:
_PROSPECT_SYSTEM_INSTRUCTION = f"""\
You are a B2B sales research assistant. You have no web search or scraping tool — you work only
from: (a) the prospect information the user gives you, and (b) general, non-specific knowledge you
already have about this kind of role.

Rules:
- Never invent specific facts (names, dates, quotes, news, projects) you were not given.
- Clearly distinguish known information from reasonable inference.
- Where something cannot be determined, use exactly "{UNKNOWN}" — do not guess.
- Be concise and factual, not speculative.
"""


def prospect_research_agent(
    prospect_name: str, job_title: str, company: str, company_website: str
) -> ProspectResearch:
    """Agent 1: builds structured prospect research from the info provided."""

    user_prompt = f"""\
Prospect information supplied by the user (no external research tool was used):

Name: {prospect_name}
Job Title: {job_title}
Company: {company}
Company Website: {company_website}

Using only the information above, plus reasonable general knowledge about this kind of role,
produce:
- role_summary: 1-3 sentence summary of their likely role and responsibilities
- relevant_context: relevant professional/company context items
- personalization_points: genuine, specific personalization angles for a sales email
- potential_pain_points: plausible business concerns for someone in this role

Do not invent specific facts you were not given. Use "{UNKNOWN}" for anything you cannot
reasonably determine.
"""

    return call_nemotron_structured(_PROSPECT_SYSTEM_INSTRUCTION, user_prompt, ProspectResearch)


## 08. Company Intelligence Agent

**Role:** understand the target company and identify plausible sales opportunities, building on
what Agent 1 already found about the prospect (so the two agents don't duplicate research).


In [ ]:
_COMPANY_SYSTEM_INSTRUCTION = f"""\
You are a B2B sales research assistant. You have no web search or scraping tool — you work only
from: (a) the company information the user gives you, (b) the prospect research already gathered,
and (c) general, non-specific knowledge you already have about this kind of company/industry.

Rules:
- Never fabricate specific company facts (funding amounts, dates, headcount, news) you were not given.
- Clearly distinguish known information from reasonable inference.
- Where something cannot be determined, use exactly "{UNKNOWN}" — do not guess.
- Be concise and factual, not speculative.
"""


def company_intelligence_agent(
    company: str, company_website: str, prospect_data: ProspectResearch
) -> CompanyIntelligence:
    """Agent 2: builds structured company intelligence, using Agent 1's output as context."""

    user_prompt = f"""\
Company information supplied by the user (no external research tool was used):

Company: {company}
Website: {company_website}

Context already known about a contact there ({prospect_data.prospect_name}, {prospect_data.job_title}):
Role summary: {prospect_data.role_summary}
Potential pain points already identified: {prospect_data.potential_pain_points}

Using only the information above, plus reasonable general knowledge about this company/industry,
produce:
- company_name: "{company}"
- industry: the company's industry, or "{UNKNOWN}"
- company_summary: 2-3 sentence summary
- products_or_services: their products/services
- target_market: description of their target customers, or "{UNKNOWN}"
- potential_challenges: plausible business challenges
- potential_use_cases: plausible use cases for a B2B product being pitched to them

Do not invent specific facts you were not given. Use "{UNKNOWN}" for anything you cannot
reasonably determine.
"""

    return call_nemotron_structured(_COMPANY_SYSTEM_INSTRUCTION, user_prompt, CompanyIntelligence)


## 09. Email Writer Agent

**Role:** write the personalized email using *only* what Agents 1 and 2 already found. This agent
does no research of its own — it just writes, so it can never introduce a fact the earlier agents
didn't establish.


In [ ]:
_EMAIL_SYSTEM_INSTRUCTION = """\
You are an experienced B2B sales rep writing a short, personalized cold outreach email.
You use ONLY the prospect research and company intelligence you are given — you do not know
anything else about the prospect or company, and you must not invent details.

Requirements for the email:
- Maximum 120 words total.
- Professional, concise, natural tone — not pushy or full of sales cliches.
- Personalized and relevant to the prospect's role.
- Relevant to the company's potential needs.
- Mention one genuinely relevant, specific detail from the research provided.
- Connect that detail to a plausible business problem, and briefly explain how the product helps.
- End with exactly one clear, low-friction call to action.
- No fake personalization, no unsupported claims, no generic spam language.
- Never mention that this email or its content was generated by AI.
"""


def email_writer_agent(
    prospect_data: ProspectResearch,
    company_data: CompanyIntelligence,
    product_description: str,
) -> SalesEmail:
    """Agent 3: writes the personalized sales email from the two prior agents' structured output.
    Does not perform any new research."""

    prompt = f"""\
PROSPECT RESEARCH
Name: {prospect_data.prospect_name}
Title: {prospect_data.job_title}
Company: {prospect_data.company}
Role summary: {prospect_data.role_summary}
Relevant context: {prospect_data.relevant_context}
Personalization points: {prospect_data.personalization_points}
Potential pain points: {prospect_data.potential_pain_points}

COMPANY INTELLIGENCE
Industry: {company_data.industry}
Company summary: {company_data.company_summary}
Products/services: {company_data.products_or_services}
Potential challenges: {company_data.potential_challenges}
Potential use cases: {company_data.potential_use_cases}

OUR PRODUCT
{product_description}

Write the sales email now as a SalesEmail JSON object with "subject" and "body" fields.
"""

    return call_nemotron_structured(_EMAIL_SYSTEM_INSTRUCTION, prompt, SalesEmail)


## 10. AI SDR Pipeline

Wires the three agents together in a fixed sequence (no agent talks to another agent directly —
this Python function is the only thing that controls the flow) and turns any failure into a clear,
per-stage error message instead of a crash. Agents run one after another, never concurrently.


In [ ]:
from typing import Optional
from pydantic import BaseModel


class PipelineResult(BaseModel):
    """Everything the UI needs, plus which stage (if any) failed."""
    prospect_data: Optional[ProspectResearch] = None
    company_data: Optional[CompanyIntelligence] = None
    email_data: Optional[SalesEmail] = None
    error_stage: Optional[str] = None
    error_message: Optional[str] = None


def run_ai_sdr(
    prospect_name: str,
    job_title: str,
    company_name: str,
    company_website: str,
    product_description: str,
) -> PipelineResult:
    """
    1. Validate inputs
    2. Prospect Research Agent
    3. Company Intelligence Agent
    4. Email Writer Agent
    5. Return results (or the first error encountered)
    Runs strictly sequentially — never concurrently.
    """
    # --- 1. Validate inputs ---
    required = {
        "Prospect Name": prospect_name,
        "Job Title": job_title,
        "Company Name": company_name,
        "Company Website": company_website,
        "Your Product": product_description,
    }
    missing = [name for name, value in required.items() if not value or not value.strip()]
    if missing:
        return PipelineResult(
            error_stage="Input Validation",
            error_message=f"Missing required field(s): {', '.join(missing)}.",
        )

    # --- 2. Prospect Research Agent ---
    try:
        print("Running Prospect Research Agent...")
        prospect_data = prospect_research_agent(prospect_name, job_title, company_name, company_website)
    except Exception as e:
        traceback.print_exc()
        return PipelineResult(
            error_stage="Prospect Research",
            error_message=f"Prospect Research Agent failed: {e}",
        )

    # --- 3. Company Intelligence Agent ---
    try:
        print("Running Company Intelligence Agent...")
        company_data = company_intelligence_agent(company_name, company_website, prospect_data)
    except Exception as e:
        traceback.print_exc()
        return PipelineResult(
            prospect_data=prospect_data,
            error_stage="Company Intelligence",
            error_message=f"Company Intelligence Agent failed: {e}",
        )

    # --- 4. Email Writer Agent ---
    try:
        print("Running Email Writer Agent...")
        email_data = email_writer_agent(prospect_data, company_data, product_description)
    except Exception as e:
        traceback.print_exc()
        return PipelineResult(
            prospect_data=prospect_data,
            company_data=company_data,
            error_stage="Email Writer",
            error_message=f"Email Writer Agent failed: {e}",
        )

    # --- 5. Done ---
    print("Pipeline complete.")
    return PipelineResult(prospect_data=prospect_data, company_data=company_data, email_data=email_data)

## 11. Gradio Interface

In [ ]:
import gradio as gr


def _bullet_list(items: list[str]) -> str:
    items = [i for i in items if i and i.strip()]
    if not items:
        return "_None found._"
    return "\n".join(f"- {item}" for item in items)


def _format_prospect(p: ProspectResearch) -> str:
    return f"""\
**Role summary:** {p.role_summary}

**Relevant context:**
{_bullet_list(p.relevant_context)}

**Personalization points:**
{_bullet_list(p.personalization_points)}

**Potential pain points:**
{_bullet_list(p.potential_pain_points)}
"""


def _format_company(c: CompanyIntelligence) -> str:
    return f"""\
**Industry:** {c.industry}

**Company summary:** {c.company_summary}

**Products/services:**
{_bullet_list(c.products_or_services)}

**Potential challenges:**
{_bullet_list(c.potential_challenges)}

**Potential use cases:**
{_bullet_list(c.potential_use_cases)}
"""


def _format_email(e: SalesEmail) -> str:
    return f"""\
**Subject:** {e.subject}

---

{e.body}
"""


def _error_md(stage: str, message: str) -> str:
    return f"### ⚠️ {stage} failed\n\n{message}"


# --- State kept between the main run and a "Regenerate Email" click ---
_last_prospect_data: Optional[ProspectResearch] = None
_last_company_data: Optional[CompanyIntelligence] = None


def on_run(prospect_name, job_title, company_name, company_website, product_description):
    global _last_prospect_data, _last_company_data

    result = run_ai_sdr(
        prospect_name, job_title, company_name, company_website, product_description
    )

    if result.error_stage:
        error_box = _error_md(result.error_stage, result.error_message)
        prospect_md = _format_prospect(result.prospect_data) if result.prospect_data else "_Not reached._"
        company_md = _format_company(result.company_data) if result.company_data else "_Not reached._"
        _last_prospect_data = result.prospect_data
        _last_company_data = result.company_data
        return prospect_md, company_md, error_box, gr.update(interactive=result.prospect_data is not None and result.company_data is not None)

    _last_prospect_data = result.prospect_data
    _last_company_data = result.company_data

    return (
        _format_prospect(result.prospect_data),
        _format_company(result.company_data),
        _format_email(result.email_data),
        gr.update(interactive=True),
    )


def on_regenerate_email(product_description):
    # Only re-runs the Email Writer Agent, reusing the existing research and company intelligence.
    if _last_prospect_data is None or _last_company_data is None:
        return _error_md("Email Writer", "Run the pipeline at least once before regenerating the email.")
    try:
        print("Regenerating email...")
        email_data = email_writer_agent(_last_prospect_data, _last_company_data, product_description)
        return _format_email(email_data)
    except Exception as e:
        traceback.print_exc()
        return _error_md("Email Writer", str(e))


with gr.Blocks(title="AI SDR") as demo:
    gr.Markdown("# 🚀 AI SDR\nThree agents (powered by NVIDIA Nemotron 3.5 Lightning) research a prospect, research their company, and draft a personalized outreach email.")

    with gr.Row():
        with gr.Column(scale=1):
            prospect_name_in = gr.Textbox(label="Prospect Name", placeholder="Alex Morgan")
            job_title_in = gr.Textbox(label="Job Title", placeholder="VP of Engineering")
            company_in = gr.Textbox(label="Company Name", placeholder="Acme Technologies")
            website_in = gr.Textbox(label="Company Website", placeholder="https://example.com")
            product_in = gr.Textbox(
                label="Your Product",
                placeholder="AI-powered developer productivity platform",
                lines=3,
            )
            run_btn = gr.Button("🚀 Run AI SDR", variant="primary")

        with gr.Column(scale=2):
            with gr.Accordion("🔎 Prospect Research", open=True):
                prospect_out = gr.Markdown()
            with gr.Accordion("🏢 Company Intelligence", open=True):
                company_out = gr.Markdown()
            with gr.Accordion("✉️ Personalized Email", open=True):
                email_out = gr.Markdown()
                regenerate_btn = gr.Button("🔄 Regenerate Email", interactive=False)

    run_btn.click(
        fn=on_run,
        inputs=[prospect_name_in, job_title_in, company_in, website_in, product_in],
        outputs=[prospect_out, company_out, email_out, regenerate_btn],
    )

    regenerate_btn.click(
        fn=on_regenerate_email,
        inputs=[product_in],
        outputs=[email_out],
    )


## 12. Demo

Runs the full pipeline once in-notebook against the sample prospect below (useful for sanity-checking
before you touch the UI), then launches the Gradio app.


In [ ]:
# --- Sample data (from the spec) ---
DEMO_PROSPECT_NAME = "Alex Morgan"
DEMO_JOB_TITLE = "VP of Engineering"
DEMO_COMPANY = "Acme Technologies"
DEMO_WEBSITE = "https://example.com"
DEMO_PRODUCT = "AI-powered developer productivity platform"

demo_result = run_ai_sdr(
    DEMO_PROSPECT_NAME, DEMO_JOB_TITLE, DEMO_COMPANY, DEMO_WEBSITE, DEMO_PRODUCT
)

if demo_result.error_stage:
    print(f"[{demo_result.error_stage}] {demo_result.error_message}")
else:
    print("=== Prospect Research ===")
    print(demo_result.prospect_data.model_dump_json(indent=2))
    print("\n=== Company Intelligence ===")
    print(demo_result.company_data.model_dump_json(indent=2))
    print("\n=== Sales Email ===")
    print(f"Subject: {demo_result.email_data.subject}\n")
    print(demo_result.email_data.body)


Running Prospect Research Agent...
Running Company Intelligence Agent...
Running Email Writer Agent...
Pipeline complete.
=== Prospect Research ===
{
  "prospect_name": "Alex Morgan",
  "job_title": "VP of Engineering",
  "company": "Acme Technologies",
  "role_summary": "Leads engineering strategy, team management, and technical delivery for a technology organization.",
  "relevant_context": [
    "Senior engineering leadership position"
  ],
  "personalization_points": [
    "Unknown / Not provided"
  ],
  "potential_pain_points": [
    "Managing engineering team productivity and resource allocation",
    "Balancing technical debt with product delivery timelines",
    "Budget and resource constraints for engineering initiatives"
  ]
}

=== Company Intelligence ===
{
  "company_name": "Acme Technologies",
  "industry": "Unknown / Not provided",
  "company_summary": "Acme Technologies is a technology organization focused on engineering strategy, team management, and technical delivery.

---**To resolve the `NameError: name 'demo' is not defined`, please ensure you have run the cell above (cell `0ff571d9`) first. This cell defines the `demo` object.**---

In [18]:
# Launch the Gradio app (opens an in-notebook UI + a shareable link in Colab)
demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8e18d674b6f3de3734.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Running Prospect Research Agent...
Running Company Intelligence Agent...
Running Email Writer Agent...
Pipeline complete.
Running Prospect Research Agent...
Running Company Intelligence Agent...
Running Email Writer Agent...
Pipeline complete.
Regenerating email...
Running Prospect Research Agent...
Running Company Intelligence Agent...
Running Email Writer Agent...
Pipeline complete.
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8e18d674b6f3de3734.gradio.live
